# Smillie colon atlas

Seven cell types of the human colon, in healthy tissue and in non-inflamed and
inflamed tissue of patients with ulcerative colitis, from
[Smillie et al., Cell 2019](https://doi.org/10.1016/j.cell.2019.06.029). Each of
the 722 profiles sums the cells of one cell type in one biopsy, and was scored
with Compass ([Wagner et al., Cell 2021](https://doi.org/10.1016/j.cell.2021.05.045)).
The profiles are built as in the
[Compass pseudobulk notebook](https://compass-wagnerlab.readthedocs.io/en/latest/notebooks/pseudobulk.html).
The notebook compares the cell types with one another, then inflamed and
non-inflamed epithelium with healthy epithelium.

It follows the server's Smillie tutorial with the library calls the server makes.
The calls notebook 01 introduces are used here as there; new here are groups from
two columns, comparison sides joined from several groups, each group against the
rest of the chosen, the analysis overview, and adjusting for read depth.

## Before you run it

Install the repository with its development dependencies, then install the
example data:

```bash
python data/download_datasets.py
```

That script fetches the same author-maintained archive the Metabis server
deploys, so the notebook and the application read byte-identical inputs.

In [ ]:
# ruff: noqa: E402, I001
import os
import sys
import tempfile
import warnings
from pathlib import Path

import pandas as pd
from bokeh.io import output_notebook, show

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    DEFAULT_LANDSCAPE_DELTA,
    AdjustmentColumn,
    AnalysisWorkspace,
    ComparisonAdjustment,
    ExperimentAnalysis,
    ExpressionData,
    ExpressionPreprocessingConfig,
    GroupComparison,
    MetabolicModel,
    OverviewCell,
    RankedSubsystem,
    RankingScore,
    ReactionData,
    ReactionPreprocessingConfig,
    SampleMetadata,
    add_category_key,
    add_map_bands,
    add_pathway_key,
    band_name,
    build_landscape,
    build_reduction_presentation,
    category_numbers,
    each_against_the_rest,
    fit_view_at_one_scale,
    highlight_reactions,
    keep_map_scale,
    map_bands,
    map_ids_for_reactions,
    maps_for_reactions,
    overview_color_bar,
    plot_analysis_overview,
    plot_comparison_ranking,
    plot_reduction_artifact,
    plot_sample_heatmap,
    plot_statistical_scatterplots,
    plot_subsystem_comparison,
    resolve_map_reactions,
    select_comparison_subsystems,
    select_shared_subsystems,
    subsystem_delta,
    subsystem_members,
    with_thresholds,
)
from metabis.analysis_workspace import RevisionKind
from metabis.version import __version__ as metabis_version
from metabis.viz.metabolic_visualization import (
    SUBSYSTEM_HUMAN1_VARIANT,
    EnrichmentConfig,
    GraphGenerationConfig,
    MetabolicGraphPipeline,
    PipelineConfig,
    RenderingConfig,
    get_available_subsystem_graph_resource_paths,
    render_graph_artifact,
)

output_notebook()

In [ ]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]
    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the Metabis repository root.")


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = repo_root / "data"
reaction_data_path = data_dir / "smillie_pseudobulk_reactions.tsv"
expression_data_path = data_dir / "smillie_pseudobulk_expression_cp10k.tsv"
metadata_path = data_dir / "smillie_pseudobulk_metadata.tsv"
required_example_files = (reaction_data_path, expression_data_path, metadata_path)
missing = [path.name for path in required_example_files if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing Smillie example files: "
        + ", ".join(missing)
        + ". Run `python data/download_datasets.py`."
    )

print(f"Metabis version: {metabis_version}")
for path in required_example_files:
    print(f"Found {path.name}")

## 1. Load data

The three files are loaded as the server loads its `smillie` example, as in
notebook 01: Compass scores computed on RECON2, one column per direction of each
reaction, read as Compass activity scores; expression already scaled per profile,
read with log1p; the sample sheet keyed by its `sample` column.

In [ ]:
model = MetabolicModel(use_cache=True)
reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    data_mode="dual",
    metabolic_model=model,
    reaction_name_format="rxnRecon3DID",
    csv_separator="\t",
    samples_in="columns",
    check_old_identifiers=True,
    preprocessing=ReactionPreprocessingConfig(compass_activity=True, zero_below=1e-6),
    source_reactions=model.get_source_reactions("RECON2") or None,
)
assert reaction_validation["status"] == "success", reaction_validation["messages"]
model.reverse_reactions(sorted(reaction_data.reversed_reactions))

reaction_df = reaction_data.get_reaction_df()
alignment_report = reaction_validation["alignment_report"]
subsystems_reached = alignment_report["subsystems_with_reactions"]
subsystems_total = alignment_report["total_subsystems"]
blocked = reaction_validation["preprocessing_report"].parameters["blocked_reactions"]
print(f"{reaction_df.shape[0]} profiles, {reaction_df.shape[1]:,} reaction columns")
print(f"Subsystems reached: {subsystems_reached} of {subsystems_total}")
print(f"Reactions blocked in every profile, removed: {blocked}")
assert reaction_df.shape == (722, 1828)
assert (subsystems_reached, subsystems_total, blocked) == (85, 154, 332)

The 722 profiles load with 1,828 reaction columns, which reach 85 of Human1's 154
subsystems; 332 reactions blocked in every profile are removed.

In [ ]:
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    gene_format="geneSymbols",
    preprocessing=ExpressionPreprocessingConfig(log1p=True),
    data_format="csv",
    csv_separator="\t",
    samples_in="columns",
)
assert expression_validation["status"] == "success", expression_validation["messages"]
expression_df = expression_data.get_expression_df()
print(f"{expression_df.shape[1]:,} metabolic genes")
assert expression_df.shape == (722, 2256)

metadata = SampleMetadata.from_table(metadata_path, sample_id_column="sample")
dm = AnalysisWorkspace(
    metabolic_model=model,
    reaction_data=reaction_data,
    expression_data=expression_data,
)
metadata_alignment = dm.update_sample_metadata(metadata)
sample_sheet = metadata.table
cell_types = sample_sheet["CellType"].nunique()
health_states = sample_sheet["Health"].nunique()
print(f"Aligned samples: {len(metadata_alignment.aligned_samples)}")
print(f"Columns: {list(sample_sheet.columns)}")
print(f"{cell_types} cell types, {health_states} health states")
assert len(metadata_alignment.aligned_samples) == 722
assert (cell_types, health_states) == (7, 3)

The expression holds 2,256 metabolic genes, and the sheet aligns to all 722
profiles. Besides `CellType` and `Health`, it holds each profile's subject,
location, number of cells (`n_cells`) and total counts (`nUMI`).

## 2. Groups and heatmap

Groups made from two columns give each combination of cell type and health state
its own base group, named `CellType | Health`. The analyses below join them when a
question needs larger sides.

In [ ]:
dm.load_groups_from_metadata(["CellType", "Health"])
group_sizes = pd.Series(
    {group: len(dm.get_group_samples(group)) for group in dm.get_group_ids()}
).sort_values(kind="mergesort")
print(f"{len(group_sizes)} groups")
print(group_sizes.to_string())
assert len(group_sizes) == 21
assert (group_sizes.index[0], group_sizes.iloc[0]) == ("Glia | Inflamed", 19)
assert (group_sizes.index[-1], group_sizes.iloc[-1]) == ("Epithelial | Healthy", 48)

The 21 groups run from 19 profiles (`Glia | Inflamed`) to 48
(`Epithelial | Healthy`).

The map is a PCA of the reaction scores, fitted as the server fits it. The heatmap
keeps the 45 most variable reactions and scales each across the profiles, gold
above the average, light gray at it and purple below, with the profiles ordered
by group under a band in its color.

In [ ]:
analysis_dependencies = frozenset(
    {
        RevisionKind.DATASET,
        RevisionKind.METADATA,
        RevisionKind.GROUPS,
        RevisionKind.PREPROCESSING,
    }
)
analysis_snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
experiment = ExperimentAnalysis(dm, auto_run=False)
experiment.calculate_pca(data_type="reactions", n_variable_features=5000)
for result in experiment.get_reduction_results():
    experiment.install_reduction_artifact(
        result.build_artifact(
            provenance=analysis_snapshot.provenance,
            source_fingerprints=dict(dm.source_fingerprints),
        )
    )
dm.install_experiment_analysis(analysis_snapshot.provenance, experiment)
dm.refresh_experiment_analysis_presentation()
reaction_pca = dm.get_reduction_artifact("PCA", "reactions")

# The profiles in group-name order: the three health states of a cell type together.
sample_groups = {
    sample: group
    for group in sorted(dm.get_group_ids())
    for sample in dm.get_group_samples(group)
}
reaction_names = {
    reaction: (model.get_reaction_reference_data(reaction) or {}).get("short_name")
    or reaction
    for reaction in reaction_df.columns
}
show(
    plot_sample_heatmap(
        reaction_df,
        sample_groups,
        count=45,
        labels=reaction_names,
        group_colors=dm.get_group_colors(),
        title="45 most variable reactions",
        width=600,
        data_type="reactions",
        label_limit=30,
    )
)

The profiles fall into blocks by cell type, and the three health states of one
cell type look alike: cell type is the larger source of variation, and health
state a smaller one within it. B-cell profiles, for example, are low across the
nucleotide kinase rows, and epithelial profiles high in the carbonate hydro-lyase
(carbonic anhydrase) rows.

A click on a carbonate hydro-lyase row colors the map by that reaction, here
MAR03955:

In [ ]:
carbonic_anhydrase = "MAR03955"
print(f"{carbonic_anhydrase}: {reaction_names[carbonic_anhydrase]}")
presentation = build_reduction_presentation(
    dm,
    reaction_pca,
    width=650,
    height=480,
    show_sample_labels=True,
    show_legend=False,
    colour_by=("reactions", carbonic_anhydrase),
)
show(plot_reduction_artifact(reaction_pca, presentation))

scores = reaction_df[carbonic_anhydrase]
standardized = (scores - scores.mean()) / scores.std()
group_means = standardized.groupby(sample_groups).mean().sort_values(ascending=False)
print("Mean standardized score by group:")
print(group_means.round(2).to_string())
epithelial = [
    f"Epithelial | {state}" for state in ("Healthy", "Non-inflamed", "Inflamed")
]
assert reaction_names[carbonic_anhydrase].startswith("carbonate hydro-lyase")
assert list(group_means.index[:3]) == epithelial

The epithelial profiles run gold and gather together, the healthy ones highest and
the inflamed ones lowest.

## 3. Healthy cell types

An analysis compares each chosen group with the other chosen groups together, as
`each_against_the_rest` sets out its comparisons; each comparison's Baseline is a
side built from several base groups. Taking the healthy profiles only keeps the
health state out of the cell types. The analyses use the server's defaults unless
a step changes them: the Wilcoxon rank-sum test, Benjamini-Hochberg correction,
and reactions and genes significant at adjusted p < 0.05. The map shows the
healthy profiles in their groups' colors, the others faded.

In [ ]:
thresholds = {"pval": 0.05, "cohens_d": None, "log2_fc": None, "mean_difference": None}
delta_config = {
    **DEFAULT_LANDSCAPE_DELTA,
    "reaction_direction_criterion": dm.reaction_direction_criterion.value,
}


def run_analysis(groups, test="wilcoxon", adjustment=None):
    """Each chosen group against the others together, by the label of its Target."""
    snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
    comparisons = {}
    for spec in each_against_the_rest([(group, (group,)) for group in groups]):
        comparison = GroupComparison(
            spec=spec, test=test, correction="fdr_bh", adjustment=adjustment
        )
        comparison.resolve_membership(dm)
        for for_genes in (False, True):
            comparison.run_test(
                dm,
                for_genes=for_genes,
                test=test,
                correction="fdr_bh",
                register_distributions=False,
            )
        artifact = comparison.build_artifact(
            provenance=snapshot.provenance,
            source_fingerprints=dict(dm.source_fingerprints),
        )
        comparisons[spec.target.label] = artifact
    return comparisons


def judged(artifact):
    """A comparison with its significance decided at ``thresholds``, to read."""
    return with_thresholds(artifact, thresholds, thresholds)


def describe(comparisons):
    for label, computed in comparisons.items():
        artifact = judged(computed)
        membership = artifact.membership
        joined = len(membership.baseline_base_group_ids)
        print(
            f"{label} ({len(membership.target_samples)} profiles) against "
            f"{membership.baseline_label} ({len(membership.baseline_samples)} "
            f"profiles, {joined} base group{'s' if joined > 1 else ''}): "
            f"{len(artifact.significant_reactions):,} reactions and "
            f"{len(artifact.significant_genes):,} genes significant"
        )


healthy_cell_types = [
    f"{cell_type} | Healthy"
    for cell_type in (
        "Epithelial",
        "B_cells",
        "T_cells",
        "Myeloid",
        "Fibroblasts",
        "Endothelial",
        "Glia",
    )
]
healthy_map = build_reduction_presentation(
    dm,
    reaction_pca,
    width=650,
    height=480,
    show_sample_labels=True,
    show_legend=False,
    highlight_group_ids=tuple(healthy_cell_types),
)
show(plot_reduction_artifact(reaction_pca, healthy_map))

cell_type_analysis = run_analysis(healthy_cell_types)
describe(cell_type_analysis)
assert len(cell_type_analysis) == 7
assert (
    len(judged(cell_type_analysis["Epithelial | Healthy"]).significant_reactions)
    == 1644
)

The analysis overview gives each subsystem's Delta score in each cell type against
the other six, from −1 (every reaction lower than in the rest) to +1 (every one
higher), for the 20 subsystems whose Delta varies most across the cell types.

In [ ]:
group_colors = dm.get_group_colors()
numbers = category_numbers(
    category
    for category, members in model.get_subsystems_by_category().items()
    if members
)


def entry(landscape, subsystem, strength=0.0):
    category = str(landscape.categories[subsystem] or "Uncategorized")
    return RankedSubsystem(
        subsystem, str(landscape.names[subsystem]), category, strength
    )


def detail(landscape, subsystem, label):
    significant = int(landscape.significant.at[subsystem, label])
    return (
        f"{significant} of {int(landscape.reactions[subsystem])} reactions significant"
    )


def overview(comparisons):
    """The reactions' Delta score of the 20 subsystems varying most, a column per group."""
    comparisons = {label: judged(artifact) for label, artifact in comparisons.items()}
    landscape = build_landscape(
        model, reaction_data, comparisons, delta_config=delta_config
    )
    rows = select_shared_subsystems([landscape], rule="spread", top=20, min_reactions=5)
    labels = list(comparisons)
    cells = {
        label: {
            row: OverviewCell(
                float(landscape.delta.at[row, label]),
                detail(landscape, row, label),
                hover=str(landscape.hovers.at[row, label]),
            )
            for row in rows
        }
        for label in labels
    }
    plot = plot_analysis_overview(
        [entry(landscape, row) for row in rows],
        labels,
        cells,
        family="reaction",
        kind="delta",
        group_colors={label: group_colors[label] for label in labels},
        against={
            label: artifact.membership.baseline_label
            for label, artifact in comparisons.items()
        },
        notes={
            label: f"{len(artifact.significant_reactions):,} significant reactions"
            for label, artifact in comparisons.items()
        },
        category_numbers=numbers,
    )
    return landscape, rows, plot


cell_type_landscape, overview_rows, overview_plot = overview(cell_type_analysis)
show(overview_plot)

shown = cell_type_landscape.delta.loc[overview_rows]
epithelium_higher = int((shown["Epithelial | Healthy"] > 0).sum())
b_cells_lower = int((shown["B_cells | Healthy"] < 0).sum())
print(f"Epithelium higher than the rest in {epithelium_higher} of {len(shown)}")
print(f"B cells lower than the rest in {b_cells_lower} of {len(shown)}")
assert epithelium_higher == b_cells_lower == len(shown) == 20

Epithelium is higher than the rest in all 20 subsystems shown, and B cells lower
in all 20, so their columns are red and blue from top to bottom.

Healthy epithelium against the other six cell types, with its 1,644 significant
reactions: the ranking keeps the 20 subsystems strongest either way, beside the
comparison's volcano plot.

In [ ]:
def view(artifact):
    """Put one comparison in view: the volcano plots and the graphs read it."""
    snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
    dm.install_group_comparison_artifact(snapshot.provenance, artifact)
    dm.refresh_group_comparison_presentation()
    for for_genes in (False, True):
        dm.apply_comparison_thresholds(
            pval_threshold=thresholds["pval"],
            cohens_d_threshold=0.0,
            log2_fc_threshold=0.0,
            mean_difference_threshold=0.0,
            for_genes=for_genes,
        )


def volcano(target, *, for_genes):
    return plot_statistical_scatterplots(
        dm,
        return_plot=True,
        show_thresholds=True,
        for_genes=for_genes,
        merge_tools=False,
        volcano="cohens_d",
        volcano_y="adjusted",
        sides=("the rest of the chosen", target),
    )


epithelium = "Epithelial | Healthy"
strongest = select_comparison_subsystems(
    {"reaction": cell_type_landscape},
    epithelium,
    rule="strongest",
    top=20,
    min_reactions=5,
)
ranking_scores = {
    row: RankingScore(
        float(cell_type_landscape.delta.at[row, epithelium]),
        detail(cell_type_landscape, row, epithelium),
        hover=str(cell_type_landscape.hovers.at[row, epithelium]),
    )
    for row in strongest
}
show(
    plot_comparison_ranking(
        [
            entry(cell_type_landscape, row, abs(ranking_scores[row].value))
            for row in strongest
        ],
        ranking_scores,
        kind="delta",
        category_numbers=numbers,
    )
)
view(cell_type_analysis[epithelium])
show(volcano(epithelium, for_genes=False))

# The subsystems the ranking reads: five reactions or more.
ranked = cell_type_landscape.reactions >= 5
epithelium_delta = cell_type_landscape.delta.loc[ranked, epithelium].rename(
    cell_type_landscape.names
)
at_one = sorted(epithelium_delta.index[epithelium_delta >= 1.0])
print(f"{len(at_one)} of {len(epithelium_delta)} subsystems at +1.0:")
print("\n".join(at_one))
read = [
    "Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism",
    "Fatty acid oxidation",
    "Carnitine shuttle (cytosolic)",
    "Carnitine shuttle (mitochondrial)",
    "Butanoate metabolism",
]
assert set(read) <= set(at_one) and len(at_one) == 30

Epithelium's strongest subsystems against the rest are at +1.0, among them the TCA
cycle, fatty acid oxidation, both carnitine shuttles and butanoate metabolism. Of
the subsystems with five reactions or more, 30 are at +1.0, and the ranking draws
the first 20.

Endothelium against the other six, in its genes:

In [ ]:
endothelium = "Endothelial | Healthy"
view(cell_type_analysis[endothelium])
show(volcano(endothelium, for_genes=True))

in_view = dm.get_group_comparison_artifact()
gene_results = in_view.get_test_results(for_genes=True)
top_gene = gene_results["cohens_d"].idxmax()
gene_symbol = model.get_gene_reference_data(top_gene)["display_label"]
gene_d = gene_results.at[top_gene, "cohens_d"]
gene_p = gene_results.at[top_gene, "adjusted_pval"]
print(f"Strongest gene: {gene_symbol}, Cohen's d {gene_d:.1f}, adjusted p {gene_p:.1e}")

reaction_results = in_view.get_test_results(for_genes=False)
top_reaction = reaction_results["cohens_d"].idxmax()
top_reaction_d = reaction_results.at[top_reaction, "cohens_d"]
top_reaction_metadata = model.get_reaction_metadata(top_reaction.removesuffix("_back"))
print(
    f"Strongest reaction: {top_reaction}, {top_reaction_metadata['name']}, "
    f"Cohen's d {top_reaction_d:.1f}"
)
print(f"Its genes: {top_reaction_metadata['gpr']}")
assert (gene_symbol, round(gene_d, 1), f"{gene_p:.1e}") == ("SLCO2A1", 11.6, "2.5e-19")
assert gene_results["adjusted_pval"].idxmin() == top_gene
assert "prostaglandin F2alpha" in top_reaction_metadata["name"]
assert "SLCO2A1" in top_reaction_metadata["gpr"].split(" or ")
assert round(top_reaction_d, 1) == 9.6

Endothelium's strongest gene against the rest is *SLCO2A1*, at the top right of
the volcano plot (Cohen's d 11.6, adjusted p 2.5 × 10<sup>−19</sup>); among the
reactions, its strongest is the prostaglandin F<sub>2α</sub> transport that Human1
assigns to *SLCO2A1* (d 9.6).

## 4. Inflamed epithelium

Inflamed against healthy epithelium: two groups make one comparison, the first
chosen against the second.

In [ ]:
inflamed_analysis = run_analysis(["Epithelial | Inflamed", "Epithelial | Healthy"])
describe(inflamed_analysis)
inflamed = inflamed_analysis["Epithelial | Inflamed"]
inflamed_judged = judged(inflamed)
inflamed_results = inflamed_judged.get_test_results(for_genes=False)
inflamed_significant = inflamed_results.loc[list(inflamed_judged.significant_reactions)]
inflamed_higher = int((inflamed_significant["cohens_d"] > 0).sum())
print(
    f"{len(inflamed_significant):,} of {len(inflamed_results):,} reactions differ, "
    f"{inflamed_higher} of them higher in inflamed epithelium"
)
assert (len(inflamed_significant), len(inflamed_results), inflamed_higher) == (
    1167,
    1828,
    591,
)

1,167 of the 1,828 reactions differ, 591 of them higher in inflamed epithelium.
Section 6 sets them out by pathway.

### Adjust for depth

Inflamed epithelial profiles hold 1.59 times as many counts as healthy ones
(median `nUMI`). The moderated t-test can compare the groups at equal depth:
`AdjustmentColumn("nUMI", log10=True)` enters log10 `nUMI` into its model beside
the group, and `ComparisonAdjustment.from_metadata` reads its values off the
sample sheet.

In [ ]:
counts = pd.to_numeric(sample_sheet["nUMI"])
depth_ratio = (
    counts[dm.get_group_samples("Epithelial | Inflamed")].median()
    / counts[dm.get_group_samples("Epithelial | Healthy")].median()
)
print(f"Median nUMI, inflamed over healthy epithelium: {depth_ratio:.2f}")
assert round(depth_ratio, 2) == 1.59


def delta_by_name(artifact):
    """Each subsystem's Delta score in one comparison, by subsystem name."""
    table = subsystem_delta(
        model, reaction_data, judged(artifact), delta_config=delta_config
    )
    return table.set_index("name")["delta"]


depth = ComparisonAdjustment.from_metadata(
    sample_sheet, [AdjustmentColumn("nUMI", log10=True)]
)
adjusted_analysis = run_analysis(
    ["Epithelial | Inflamed", "Epithelial | Healthy"],
    test="moderated_t",
    adjustment=depth,
)
adjusted = adjusted_analysis["Epithelial | Inflamed"]
fatty_acid_oxidation = {
    "adjusted": delta_by_name(adjusted)["Fatty acid oxidation"],
    "not adjusted": delta_by_name(inflamed)["Fatty acid oxidation"],
}
print(
    f"{adjusted.test}, adjusted for {', '.join(adjusted.adjusted_for)}: "
    f"{len(judged(adjusted).significant_reactions):,} reactions differ"
)
for name, value in fatty_acid_oxidation.items():
    print(f"Fatty acid oxidation, {name}: Delta {value:+.2f}")
assert len(judged(adjusted).significant_reactions) == 1109
assert [round(value, 2) for value in fatty_acid_oxidation.values()] == [-0.92, -0.90]

With depth in the model 1,109 reactions differ, and fatty acid oxidation still
scores lower (Delta −0.92, against −0.90).

## 5. Three epithelial states

Whether tissue that is not inflamed shares the response: first the three
epithelial states side by side, each against the other two, then non-inflamed
against healthy epithelium alone. The overview opens on the 20 subsystems varying
most across the three states.

In [ ]:
state_analysis = run_analysis(
    ["Epithelial | Healthy", "Epithelial | Non-inflamed", "Epithelial | Inflamed"]
)
describe(state_analysis)
state_landscape, _state_rows, state_plot = overview(state_analysis)
show(state_plot)

carnitine_shuttle = "Carnitine shuttle (mitochondrial)"
state_delta = state_landscape.delta.rename(index=state_landscape.names)
print(state_delta.loc[["Fatty acid oxidation", carnitine_shuttle]].round(2).to_string())
state_significant = {
    label: len(judged(artifact).significant_reactions)
    for label, artifact in state_analysis.items()
}
assert state_significant == {
    "Epithelial | Healthy": 1090,
    "Epithelial | Non-inflamed": 20,
    "Epithelial | Inflamed": 912,
}
assert round(state_delta.at["Fatty acid oxidation", "Epithelial | Healthy"], 2) == 0.91
assert round(state_delta.at[carnitine_shuttle, "Epithelial | Healthy"], 2) == 1.00
assert (
    round(state_delta.at["Fatty acid oxidation", "Epithelial | Inflamed"], 2) == -0.84
)
assert round(state_delta.at[carnitine_shuttle, "Epithelial | Inflamed"], 2) == -0.95

Healthy epithelium scores higher than the other two states in fatty acid
oxidation (Delta +0.91) and the mitochondrial carnitine shuttle (+1.00), and
inflamed epithelium lower (−0.84 and −0.95). Non-inflamed epithelium sits between them: only 20 of its
reactions differ from the other two together, against 1,090 for healthy and 912
for inflamed epithelium.

Non-inflamed against healthy epithelium:

In [ ]:
non_inflamed_analysis = run_analysis(
    ["Epithelial | Non-inflamed", "Epithelial | Healthy"]
)
describe(non_inflamed_analysis)
non_inflamed = non_inflamed_analysis["Epithelial | Non-inflamed"]
against_healthy = pd.DataFrame(
    {
        "Non-inflamed": delta_by_name(non_inflamed),
        "Inflamed": delta_by_name(inflamed),
    }
).loc[["Fatty acid oxidation", "Butanoate metabolism"]]
print(against_healthy.round(2).to_string())
assert len(judged(non_inflamed).significant_reactions) == 857
assert against_healthy.round(2).values.tolist() == [[-0.84, -0.90], [-0.83, -0.83]]

857 reactions differ. Non-inflamed epithelium also scores lower in fatty acid
oxidation (Delta −0.84) and butanoate metabolism (−0.83) against healthy
epithelium, as inflamed epithelium does (−0.90 and −0.83).

## 6. Read it on the maps

The inflamed epithelium comparison is put in view, and its Delta scores color the
metabolic overview, as in notebook 01: each subsystem by its Delta score, blue
where inflamed epithelium is lower, with the ranking listing the subsystems by
Delta score within each category.

In [ ]:
view(inflamed)
dm.subsystem_analysis.update_delta_config(delta_config)
subsystem_stats = dm.subsystem_analysis.rank_subsystems_by_delta_score(for_genes=False)

# The server's hover detail: full on the overview, medium on a subsystem's graph.
full_hover = {
    "show_hover_stats": True,
    "show_hover_distribution_plots": True,
    "show_hover_color_reference": True,
}
medium_hover = {**full_hover, "show_hover_distribution_plots": False}


def draw(element, graph_type, *, use_human1_map, layout_engine, hover):
    """A graph of the comparison in view, colored by Cohen's d and Delta score."""
    snapshot = dm.capture_analysis_snapshot(analysis_dependencies)
    pipeline = MetabolicGraphPipeline(
        model,
        PipelineConfig(
            generation_config=GraphGenerationConfig(
                use_human1_map=use_human1_map,
                layout_engine=layout_engine,
                resource_registry=model.resource_registry,
            ),
            enrichment_config=EnrichmentConfig(
                color_metric="cohens_d", delta_options=delta_config, **hover
            ),
            show_plot=False,
        ),
    )
    return pipeline.build_artifact(
        element,
        graph_type,
        data_manager=dm,
        provenance=snapshot.provenance,
        source_fingerprints=dict(dm.source_fingerprints),
    )


network = draw(
    None,
    "metabolic_network",
    use_human1_map=True,
    layout_engine="fdp",
    hover=full_hover,
)
# Room for the network and, below it, its color key and the category key.
network_rendering = RenderingConfig(
    width=975,
    height=950,
    sizing_mode="stretch_width",
    aspect_ratio=None,
    zoom=1.5,
    match_aspect=True,
)
graph_rendering = RenderingConfig(
    width=975,
    height=900,
    sizing_mode="stretch_width",
    aspect_ratio=None,
    match_aspect=True,
    fit_view_to_bounds=True,
)
network_plot = render_graph_artifact(network, network_rendering).plot
# Across, below the network, as the app's overview sets it.
color_bar = overview_color_bar(
    dm,
    visualization_mode="group_comparison",
    delta_options=delta_config,
)
color_bar.update(orientation="horizontal", width=200, height=10, location="center")
network_plot.add_layout(color_bar, "below")
add_category_key(network_plot, visible=True)
# At one scale in whatever room the keys leave, as the app's overview.
show(fit_view_at_one_scale(network_plot))
show(
    plot_subsystem_comparison(
        dm,
        subsystem_stats=subsystem_stats,
        metric="cohens_d",
        max_points_per_subsystem=50,
        return_plot=True,
        delta_options=delta_config,
    )
)

In [ ]:
members = subsystem_members(
    model, inflamed_judged, criterion=dm.reaction_direction_criterion
)
subsystem_names = {
    subsystem: model.get_subsystem_name(subsystem)
    for subsystem in subsystem_stats["subsystem"]
}
overview_delta = subsystem_stats.set_index("subsystem")["delta_score"].rename(
    subsystem_names
)


def directions(reactions):
    """How many of ``reactions`` are significantly lower and higher in the Target."""
    rows = inflamed_results.loc[list(reactions)]
    rows = rows[rows["is_significant"]]
    return int((rows["cohens_d"] < 0).sum()), int((rows["cohens_d"] > 0).sum())


fao_lower, fao_higher = directions(members["fatty_acid_oxidation"])
print(
    f"Fatty acid oxidation: Delta {overview_delta['Fatty acid oxidation']:+.2f}, "
    f"{fao_lower} of {len(members['fatty_acid_oxidation'])} reactions lower, "
    f"{fao_higher} higher"
)
pathways = [carnitine_shuttle, "Butanoate metabolism", "Glycolysis / Gluconeogenesis"]
for name in pathways:
    print(f"{name}: Delta {overview_delta[name]:+.2f}")
assert round(overview_delta["Fatty acid oxidation"], 2) == -0.90
assert (fao_lower, len(members["fatty_acid_oxidation"]), fao_higher) == (158, 174, 1)
assert [round(overview_delta[name], 2) for name in pathways] == [-0.95, -0.83, 0.09]

Fatty acid oxidation has a Delta of −0.90, with 158 of its 174 reactions lower in
inflamed epithelium and one higher; the mitochondrial carnitine shuttle −0.95 and
butanoate metabolism −0.83. Glycolysis barely moves (+0.09).

The graph of Butanoate metabolism colors each reaction by its Cohen's d where
significant, and each gene by its own test:

In [ ]:
butanoate = "butanoate_metabolism"
layouts = get_available_subsystem_graph_resource_paths(
    butanoate, model.resource_registry.get_graph_base_paths(required=False)
)
butanoate_graph = draw(
    "Butanoate metabolism",
    "subsystem",
    use_human1_map=SUBSYSTEM_HUMAN1_VARIANT in layouts,
    layout_engine="dot",
    hover=medium_hover,
)
butanoate_plot = render_graph_artifact(butanoate_graph, graph_rendering).plot
add_pathway_key(butanoate_plot, dm, visualization_mode="group_comparison")
# Whole and at one scale above its key, as the app draws it.
keep_map_scale(butanoate_plot, fit_whole=True)
show(butanoate_plot)

measured = inflamed_results.loc[
    list(members[butanoate]), ["cohens_d", "adjusted_pval", "is_significant"]
]
measured.insert(
    0,
    "name",
    [
        model.get_reaction_metadata(item.removesuffix("_back"))["name"]
        for item in measured.index
    ],
)
print(measured.round({"cohens_d": 2}).to_string())
butanoate_lower, butanoate_higher = directions(measured.index)
print(
    f"{butanoate_lower} of {len(measured)} measured reactions lower, {butanoate_higher} higher"
)
assert (butanoate_lower, len(measured), butanoate_higher) == (5, 6, 0)
assert measured["cohens_d"].round(2).loc[
    ["MAR03163", "MAR07709", "MAR01434_back", "MAR04461"]
].tolist() == [-0.92, -1.05, -1.28, -0.52]

decreases = inflamed_significant[inflamed_significant["cohens_d"] < 0]["cohens_d"]
butyrate_transport = "MAR00097"
transport_d = decreases[butyrate_transport]
transport_rank = int((decreases < transport_d).sum()) + 1
print(
    f"{butyrate_transport}, {model.get_reaction_metadata(butyrate_transport)['name']} "
    f"({', '.join(model.get_reaction_metadata(butyrate_transport)['subsystems'])}): "
    f"Cohen's d {transport_d:.2f}, rank {transport_rank} of {len(decreases)} decreases"
)
assert round(transport_d, 2) == -1.57 and transport_rank <= 10

Five of its six measured reactions are lower in inflamed epithelium: the
mitochondrial oxidation of butyryl-CoA by short-chain acyl-CoA dehydrogenases
(Cohen's d −0.92) and the next hydration step (−1.05), the thiolase step between
acetoacetyl-CoA and acetyl-CoA (−1.28), and 3-hydroxybutyrate dehydrogenase
(−0.52). Outside this subsystem, butyrate's transport into mitochondria (−1.57) is
among the comparison's largest decreases.

Its compartment maps, the Metabolic Atlas maps that draw its reactions, show them
among the reactions around them:

In [ ]:
butanoate_reactions = model.get_reactions_for_subsystem(butanoate)
butanoate_maps = [
    (atlas_map.name, count)
    for atlas_map, count in maps_for_reactions(model, butanoate_reactions)
]
for name, count in butanoate_maps:
    print(f"{name}, {count} of its reactions")
assert ("Mitochondria", 6) in butanoate_maps

The Mitochondria map draws 6 of its reactions. Drawn with Butanoate metabolism
highlighted, its band glows yellow and the view frames its reactions; the map is
colored as the pathway graphs.

In [ ]:
# Light on a map, as the server draws it: statistics only.
light_hover = {**medium_hover, "show_hover_color_reference": False}
mitochondria = "mitochondria"
mitochondria_map = draw(
    mitochondria,
    "compartment",
    use_human1_map=True,
    layout_engine="dot",
    hover=light_hover,
)
map_plot = render_graph_artifact(mitochondria_map, graph_rendering).plot
map_plot.title.text = "Mitochondria map, Butanoate metabolism in view"
add_pathway_key(map_plot, dm, visualization_mode="group_comparison")
add_map_bands(
    map_plot, mitochondria, band_name(map_bands(mitochondria), "Butanoate metabolism")
)
highlight_reactions(
    map_plot, map_ids_for_reactions(model, mitochondria, butanoate_reactions)
)
keep_map_scale(map_plot, fit_whole=True)
show(map_plot)

drawn = set(resolve_map_reactions(model, mitochondria).values())
on_map = {}
for subsystem in ("fatty_acid_oxidation", "carnitine_shuttle_mitochondrial"):
    reactions = [
        item for item in members[subsystem] if item.removesuffix("_back") in drawn
    ]
    lower, higher = directions(reactions)
    on_map[subsystem] = (lower, len(reactions), higher)
    print(
        f"{model.get_subsystem_name(subsystem)} on the map: {lower} of "
        f"{len(reactions)} measured reactions lower, {higher} higher"
    )
assert on_map == {
    "fatty_acid_oxidation": (127, 129, 0),
    "carnitine_shuttle_mitochondrial": (21, 22, 0),
}

On this map, 127 of fatty acid oxidation's 129 measured reactions score lower in
inflamed epithelium and none higher, as do 21 of the carnitine shuttle's 22.